# Similarity and Record Linkage

## Objective
Compare Bloom filters and identify matches.

In [25]:
import pandas as pd

df_A = pd.read_pickle("../data/processed/df_A_blocked.pkl")
df_B = pd.read_pickle("../data/processed/df_B_blocked.pkl")

### Dice similarity

In [26]:
# Calculate similarity score
def dice_similarity(bf1, bf2):
    intersection = (bf1 & bf2).count()
    return 2 * intersection / (bf1.count() + bf2.count() + 1e-10) # 0.0000000001 to prevent division by zero

### PRE-EXTRACT DATA (CRITICAL OPTIMIZATION)

In [27]:
bloom_A = df_A["bloom"].to_dict()
bloom_B = df_B["bloom"].to_dict()
id_A_map = df_A["id"].to_dict()
id_B_map = df_B["id"].to_dict()

### Generating pairs

In [28]:
def generate_pairs(df_A, df_B, block):
    pairs = set()
    
    for val in df_A[block].dropna().unique(): # Get unique values from the blocking column.
        subA_idx = df_A[df_A[block] == val].index
        subB_idx = df_B[df_B[block] == val].index

        for i in subA_idx:
            for j in subB_idx:
                pairs.add((i, j))
    return pairs

print("\nGenerating candidate pairs using blocking...")

pairs = set()
for block_col in ["block1", "block2", "block3"]:# Trying different blocking columns (we dont want to miss potential matches)
    block_pairs = generate_pairs(df_A, df_B, block_col)
    print(f"{block_col}: {len(block_pairs)} pairs")
    pairs.update(block_pairs)

print(f"\nTotal unique candidate pairs: {len(pairs)}")

# matches = []
# for i, j in pairs: # Scoring and filtering matches.
#     sim = dice_similarity(df_A.loc[i,"bloom"], df_B.loc[j,"bloom"])
#     if sim >= 0.85: # If similarity is ≥ 85% consider it a match 
#         matches.append({
#             "id_A": df_A.loc[i,"id"],
#             "id_B": df_B.loc[j,"id"],
#             "sim": sim
#         })


Generating candidate pairs using blocking...
block1: 137632 pairs
block2: 197832 pairs
block3: 24365 pairs

Total unique candidate pairs: 329444


### Generating pairs - updated version

In [29]:
from joblib import Parallel, delayed



def compare_pair(i, j):
    # bf1 = df_A.loc[i, "bloom"]
    # bf2 = df_B.loc[j, "bloom"]
    # sim = dice_similarity(bf1, bf2)
    sim = dice_similarity(bloom_A[i], bloom_B[j])

    return (i, j, sim)


In [30]:
# Uses all CPUs
def parallel_compare(pairs, threshold=0.85):

    results = Parallel(
        n_jobs=-1,              # Use all CPUs
        batch_size=1000,        # Prevents memory overload 
        backend="loky"
        )(  
        delayed(compare_pair)(i, j)    # Each CPU runs compare_pair() independently
        for i, j in pairs
    )

    '''
    Execution flow: pairs → split → multiple CPUs → compute similarities → combine results
    '''

    print("Filtering matches...")

    # Filter matches
    matches = [
        {
            "id_A": id_A_map[i],
            "id_B": id_B_map[j],
            "similarity": sim
        }
        for i, j, sim in results 
        if sim >= threshold
    ]

    return matches

### LINKAGE

In [31]:
THRESHOLD = 0.85

matches = parallel_compare(pairs, threshold=THRESHOLD)
matches_df = pd.DataFrame(matches)

print(f"\nTotal matches found: {len(matches_df)}")

output_path = "../data/processed/matches_optimized.csv"

matches_df.to_csv(output_path, index=False)

print(f"\nMatches saved to: {output_path}")

Filtering matches...

Total matches found: 809

Matches saved to: ../data/processed/matches_optimized.csv


In [32]:
print("\nSample matches:")
print(matches_df.head())


Sample matches:
   id_A  id_B  similarity
0   220   220    0.883117
1   520  1082    0.865169
2   417   417    1.000000
3   298   298    0.892655
4   743   678    0.867052
